In [1]:
import numpy as np
import torch

from math import e 

In [2]:
# 1. 입력값 X와 정답 y 준비

# X: 입력값으로, 사람의 키(cm)를 사용
# np.array([...])는 여러 숫자를 하나의 NumPy 배열로 묶는 것.

X = np.array([169, 179, 189, 189])

# y: 정답값으로, 0는 농구선수 아님, 1은 농구선수임을 나타냄
# x와 y는 순서대로 짝지어짐. 즉, 키 160cm -> 정답 0, 키 190cm -> 정답 1
y = np.array([0, 0, 1, 1])

print('입력값 X:', X)
print('정답값 y:', y)

입력값 X: [169 179 189 189]
정답값 y: [0 0 1 1]


In [3]:
# 2. 입력값 정규화

# 평균과 표준편차를 계산
# 평균: 데이터의 중심(가운데쯤 되는 값)
# 표준편차: 데이터가 평균에서 얼마나 넓게 퍼져 있는지를 나타내는 값
X_mean = np.mean(X)
X_std = np.std(X)

# 정규화 공식: (원본값 - 평균) / 표준편차
# 입력값의 범위를 0 근처로 비슷하게 맞추면 학습이 더 안정적으로 진행됨
# 주의: 실제 학습에는 원래 키 X가 아니라, 정규화된 입력값 X_norm을 사용
#       (X_mean, X_std는 나중에 '새 입력값 예측'에서도 똑같이 다시 씀)
X_norm = (X - X_mean) / X_std

print('입력값 평균 X_mean:', X_mean)
print('입력값 표준편차 X_std:', X_std)
print('정규화된 입력값 X_norm:', X_norm)

입력값 평균 X_mean: 181.5
입력값 표준편차 X_std: 8.2915619758885
정규화된 입력값 X_norm: [-1.50755672 -0.30151134  0.90453403  0.90453403]


In [4]:
# 2-1. X_norm과 y를 PyTorch tensor로 변환

# PyTorch의 자동 미분은 'tensor'라는 자료형 위에서 동작
# tensor란, NumPy 배열과 거의 똑같이 생긴 PyTorch 전용 숫자 묶음
# 단, tensor은 '어떻게 계산되었는지'를 기억할 수 있어서 자동 미분이 가능

# 그래서 학습에 사용할 입력값(X_norm)과 정답(y)을 먼저 tensor로 바꿔 둠

# 주의(헷갈리기 쉬움):
#  X      = 원래 키(cm)
#  X_norm = 정규화된 입력값 <- 학습에는 이 값을 사용
# 따라서 아래에서도 X가 아니라 X_norm을 tensor로 변환

# dtype=torch.float32 : 소수점 계산(미분)을 위해 실수(float) 형식으로 만듦
X_norm_tensor = torch.tensor(X_norm, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)

print('학습용 입력 tensor X_norm_tensor:', X_norm_tensor)
print('학습용 정답 tensor y_tensor', y_tensor)

학습용 입력 tensor X_norm_tensor: tensor([-1.5076, -0.3015,  0.9045,  0.9045])
학습용 정답 tensor y_tensor tensor([0., 0., 1., 1.])


In [13]:
# 3. 가중치 a와 편향 b 초기값 설정 (requires_grad=True인 tensor)

# a는 가중치(weight), b는 편향(bias)
# a는 원래 키(cm)가 아니라 정규화된 입력값 X_norm에 곱해지는 값

# requires_grad=True는 PyTorch에게
# "이 값에 대한 미분값(기울기)을 계산해 줘"라고 알려주는 설정

# 여기서 a와 b는 학습으로 계속 수정해 나갈 값이므로 requires_grad=True를 설정
# (반대로 입력 X_norm_tensor와 정답 y_tensor에는 이 설정을 하지 않음
#   왜냐하면 그 둘은 학습으로 바꾸는 값이 아니기 때문)
a = torch.tensor(0.1, dtype=torch.float32, requires_grad=True)
b = torch.tensor(0.0, dtype=torch.float32, requires_grad=True)

# a, b는 tensor라서 그냥 출력하면 tensor(...) 형태로 나옴
# .item()으로 숫자만 깔끔하게 꺼내서 출력
print('초기 가중치 a:', a.item())
print('초기 편향 b:', b.item())

초기 가중치 a: 0.10000000149011612
초기 편향 b: 0.0


In [14]:
# 4. 시그모이드 함수 정의 (학습 전 NumPy 확인용)

# 시그모이드 함수는 선형 계산값 H(x)를 0~1 사이의 예측 확률 z로 변환
# 아래 함수는 '학습 전 예측/Cost 확인'(NumPy 흐름)에서만 사용
# 실제 학습 루프 안에서는 PyTorch의 torch.sigmoid를 사용
def sigmoid(H):
    return 1 / (1 + e**(-H))

In [15]:
# 5. 학습 전 예측 확인 (기존 NumPy 흐름 그대로 유지)

# H(x) = a * X_norm + b
# H(x)는 sigmoid에 들어가기 전의 선형 계산값 (확률이 아님)
# 여기서는 학습 전 상태를 눈으로 확인하려고 NumPy로 잠깐 계산해 봄

# a, b는 tensor이고 X_norm은 NumPy 배열
# 학습 전 '확인용' 계산이므로, a, b에서 .item()으로 숫자만 꺼내 NumPy와 함께 계산
# (이 셀은 학습이 아니므로 미분 그래프와 연결될 필요가 없음)
H = a.item() * X_norm + b.item()

# z = sigmoid(H)
# z는 농구선수일 예측 확률 (0~1 사이) <- H(x)가 아니라 z가 확률
z = sigmoid(H)

print('학습 전 선형 계산값 H(x):', H)
print('학습 전 시그모이드 예측 확률 z:', z)

학습 전 선형 계산값 H(x): [-0.15075567 -0.03015113  0.0904534   0.0904534 ]
학습 전 시그모이드 예측 확률 z: [0.4623823  0.49246279 0.52259795 0.52259795]


In [16]:
# 6. 학습 전 비용(Cost) 계산 (기존 NumPy 흐름 그대로 유지)

# Binary Cross Entropy(BCE) 비용 함수 (BCE = 이진 교차 엔트로피)
# 흐름: 현재 a, b로 H(x)를 구하고 -> sigmoid로 z를 구하고 -> z와 y를 비교해 Cost를 계산

# 실제값 y가 1이면 z가 1에 가까울수록 Cost가 작아지고, 
# 실제값 y가 0이면 z가 0에 가까울수록 Cost가 작아짐
# 즉, 예측 z가 정답 y에 가까울수록 Cost가 작아지고, 멀수록 커짐

# Cost = -{y * log(z) + (1-y) * log(1-z)}

# log(0)을 방지하기 위해 z가 정확히 0 또는 1이 되지 않도록 제한
epsilon = 1e-7
z_safe = np.clip(z, epsilon, 1-epsilon)

costs = -(y * np.log(z_safe) + (1-y) * np.log(1-z_safe))
mean_cost = np.mean(costs)

print('학습 전 각 샘플의 비용(Cost):', costs)
print('학습 전 평균 비용(Cost):', mean_cost)

학습 전 각 샘플의 비용(Cost): [0.62060757 0.67818525 0.64894286 0.64894286]
학습 전 평균 비용(Cost): 0.6491696313806095


In [9]:
# 7. 학습 설정 (기존 흐름 그대로 유지)

# learning_rate(학습률)는 한 번에 a, b를 얼마나 크게 수정할지 정하는 값
#   - 너무 크면: 한 번에 너무 많이 움직여 학습이 출렁이거나 발산할 수 있음
#   - 너무 작으면: 너무 조금씩 움직여 학습이 매우 느려짐
learning_rate = 0.1

# epochs(에폭)는 전체 데이터를 몇 번 반복해서 학습할지 정하는 값
# 여기서는 같은 데이터로 1000번 반복 학습
epochs = 1000

In [10]:
# 8. 경사 하강법으로 학습 (미분 계산만 PyTorch autograd로 대체)

# 한 번의 epoch(반복)에서 일어나는 단계:
#   1단계: H(x) = a*X_norm + b 계산
#   2단계: z = sigmoid(H) 계산
#   3단계: BCE Cost 계산
#   4단계: backward()로 Cost를 a, b에 대해 미분
#   5단계: a.grad, b.grad를 grad_a, grad_b로 저장 (출력용)
#   6단계: Gradient Descent로 a, b 업데이트
#   7단계: 다음 epoch를 위해 gradient 초기화
#   8단계: 학습 상태 출력

for epoch in range(epochs):
    
    # ---- 1단계: 선형 계산값 H(x) ----
    # H(x) = a * X_norm + b
    # H(x)는 sigmoid에 들어가기 전의 선형 계산값
    # H(x)는 확률이 아님 (확률은 다음 단계의 z)
    # 여기서는 학습용이므로 tensor인 a, b와 X_norm_tensor를 그대로 곱함
    # (.item()을 쓰지 않음. 그래야 계산 그래프가 연결되어 자동 미분이 가능)
    H = a * X_norm_tensor + b
    
    
    # ---- 2단계: 예측 확률 z ----
    # z는 H(x)를 sigmoid 함수에 넣어 얻은 예측 확률
    # z는 0과 1 사이의 값
    # z가 1에 가까우면 1(농구선수)일 가능성이 높고,
    # z가 0에 가까우면 0(농구선수 아님)일 가능성이 높다고 해석
    z = torch.sigmoid(H)
    
    
    # ---- 3단계: BCE Cost 직접 계산 ----
    # torch.clamp()는 log(0)을 방지하기 위한 안전장치
    # z가 정확히 0 또는 1이 되면 log(0)이 되어 계산이 깨지므로,
    # z를 epsilon ~ (1 - epsilon) 사이로 살짝 제한
    # (수학 교재의 핵심 미분 흐름은 clamp를 제외한 기본 BCE + sigmoid 기준이며,
    #   그 핵심 결과는 dC/cH = z - y임. clamp는 수치 안정성용 안전장치)
    z_safe = torch.clamp(z, epsilon, 1 - epsilon)

    # Cost = -{y * log(z) + (1-y) * log(1-z)}
    # torch.mean()으로 4개 데이터의 Cost를 평균 내 하나의 평균 Cost를 만듦
    costs = -(y_tensor * torch.log(z_safe) + (1-y_tensor) * torch.log(1-z_safe)) 
    mean_cost = torch.mean(costs)
    
    
    # ---- 4단계: backward()로 자동 미분 ----
    # backward()는 평균 Cost에서 출발해 a, b까지 연결된 계산 과정(계산 그래프)을
    # 거꾸로 따라가며 미분값(기울기)을 계산
    
    # 계산이 끝나면
    #   a.grad에는 Cost를 a로 미분한 값이,
    #   b.grad에는 Cost를 b로 미분한 값이 들어감
    
    # 수학적으로는 다음과 같음
    #   grad_a = 평균((z-y) * X_norm)  <- a.grad가 이 값  
    #   grad_b = 평균(z-y)             <- b.grad가 이 값   
    # 하지만 이 식을 직접 코드로 적지 않아도,
    # backward()가 계산 그래프를 따라 같은 값을 자동으로 구해 줌
    mean_cost.backward()
    
    
    # ---- 5단계: a.grad, b.grad를 grad_a, grad_b로 저장 (출력용) ----
    # a.grad, b.grad는 tensor임. 출력/기록할 때는 .item()으로
    # 일반 숫자처럼 꺼내는 것이 초보자에게 이해하기 쉬움
    
    # a.grad.item()은 기존 수식의 grad_a와 같은 의미이고,
    # b.grad.item()은 기존 수식의 grad_b와 같은 의미
    # (즉, a.grad가 곧 grad_a 역할. 여기서는 출력 편의를 위해 숫자로만 꺼내 둠)
    
    # 주의: 7단계에서 gradient를 0으로 초기화하므로,
    #       반드시 초기화 '전에' 이렇게 값을 저장해 둬야 함
    grad_a = a.grad.item()
    grad_b = b.grad.item()
    
    # ---- 6단계: 경사 하강법으로 a, b 업데이트 ----
    # a, b를 직접 수정하는 작업은 미분 계산 기록(계산 그래프)에 포함되면 안 됨
    # 우리가 원하는 건 '현재 Cost 기준으로 구한 a.grad, b.grad로 a, b를 고치는 것'뿐이고,
    # 이 수정 동작 자체까지 미분 대상으로 기록할 필요는 없기 때문
    # 그래서 torch.no_grad() 안에서 업데이트함
    
    # 이 부분은 기존 Gradient Descent 식과 같음
    #   a = a - learning_rate * grad_a   (여기서 a.grad가 grad_a 역할)
    #   b = b - learning_rate * grad_b   (여기서 b.grad가 grad_b 역할)
    # 즉 Cost가 줄어드는 방향(기울기 반대 방향)으로 a, b를 조금씩 움직임
    with torch.no_grad():
        a -= learning_rate * a.grad
        b -= learning_rate * b.grad
        
    # ---- 7단계: 다음 epoch를 위해 gradient 초기화 ----
    # PyTorch는 기본적으로 미분값을 덮어쓰지 않고 계속 '누적(더하기)'함
    # 즉 backward()를 또 부르면 기존 grad 위에 새 grad가 더해짐
    # 따라서 다음 epoch에서 '새로 계산한 미분값만' 사용하려면
    # 기존 gradient를 0으로 초기화해야 함
    # zero_()에서 끝의 _는 새 tensor를 만드는 게 아니라 기존 tensor 값을 직접 0으로 바꾼다는 뜻
    a.grad.zero_()
    b.grad.zero_()
    
    # ---- 8단계: 학습 상태 출력 ----
    # tensor가 그대로 출력되지 않도록 .item()으로 숫자만 꺼내 출력
    # 100 epoch마다 한 번씩, 그리고 마지막 epoch에서 출력
    if epoch % 100 == 0 or epoch == epochs-1:
        print(
            f'epoch={epoch}, '
            f'Cost={mean_cost.item():.6f}, '
            f'grad_a: {grad_a:.6f}, '
            f'grad_b: {grad_b:.6f}, '
            f'a={a.item():.6f}, '
            f'b={b.item():.6f}'
        )

epoch=0, Cost=0.649170, grad_a: -0.427301, grad_b: 0.000010, a=0.142730, b=-0.000001
epoch=100, Cost=0.185918, grad_a: -0.104153, grad_b: 0.018129, a=2.090428, b=-0.107575
epoch=200, Cost=0.118100, grad_a: -0.062464, grad_b: 0.017270, a=2.881634, b=-0.290043
epoch=300, Cost=0.087113, grad_a: -0.045685, grad_b: 0.014161, a=3.412171, b=-0.446873
epoch=400, Cost=0.068903, grad_a: -0.036218, grad_b: 0.011615, a=3.817455, b=-0.575024
epoch=500, Cost=0.056884, grad_a: -0.030035, grad_b: 0.009719, a=4.146477, b=-0.681126
epoch=600, Cost=0.048364, grad_a: -0.025652, grad_b: 0.008303, a=4.423545, b=-0.770837
epoch=700, Cost=0.042020, grad_a: -0.022376, grad_b: 0.007222, a=4.662786, b=-0.848180
epoch=800, Cost=0.037117, grad_a: -0.019834, grad_b: 0.006377, a=4.873209, b=-0.915970
epoch=900, Cost=0.033220, grad_a: -0.017804, grad_b: 0.005701, a=5.060940, b=-0.976206
epoch=999, Cost=0.030079, grad_a: -0.016161, grad_b: 0.005155, a=5.228730, b=-1.029828


In [11]:
# 9. 학습 완료 후 최종 가중치와 편향 확인

# 학습된 a와 b는 정규화된 입력값 X_norm을 기준으로 학습된 값
# (원래 키 X 기준이 아니라 X_norm 기준이라는 점!)
# a, b는 tensor이므로 .item()으로 숫자만 꺼내서 출력
print(f'\n학습 완료 후의 최적값: a = {a.item()}, b = {b.item()}')


학습 완료 후의 최적값: a = 5.228729724884033, b = -1.0298280715942383


In [12]:
# 10. 새로운 입력값 예측

# 키가 185cm인 사람이 농구선수인지 예측
input_height = 185

# 새로운 입력값도 학습 데이터와 '같은 기준'으로 정규화해야 함
# 학습 때 사용한 X_mean, X_std를 그대로 다시 사용 (새로 계산하면 안 됨)
input_norm = (input_height - X_mean) / X_std

# 예측은 학습이 아니므로 a, b를 업데이트하지 않음
# 따라서 미분 계산 기록도 필요 없으므로 with torch.no_grad() 안에서 계산
with torch.no_grad():
    # 학습된 a, b는 tensor이므로, 입력값도 tensor로 맞춰서 계산
    input_norm_tensor = torch.tensor(input_norm, dtype=torch.float32)
    # H(x) = a * X_norm + b (선형 계산값 - 확률이 아님)
    H_input = a * input_norm_tensor + b
    # z = sigmoid(H) (예측 확률 - 0~1 사이)
    probability = torch.sigmoid(H_input)

print(f'키가 {input_height}cm인 사람이 농구선수일 확률(z): {probability.item():.4f}')
    
# 0.5 이상이면 1(농구선수), 미만이면 0(농구선수 아님)
pred = 1 if probability.item() >= 0.5 else 0

if pred == 1:
    print('판별 결과: 농구선수입니다.')
else:
    print('판별 결과: 농구선수가 아닙니다.')

키가 185cm인 사람이 농구선수일 확률(z): 0.7645
판별 결과: 농구선수입니다.
